In [ ]:
So chains were definitely a big improvement.
They made our code cleaner, reduced repetition, and allowed us to connect multiple components into a pipeline.
But as developers started building more complex application, new problem started to appear.
there wasn't just one type of chain.
Langchain had introduced many different chains for different use cases.
for ex:
    .LLMChain --> for simple prompt + model tasks
    .SequentialChain----> for multiple workflow
    .RouterChain -----> for dyanamic decision making
    .RetrievalQA Chain ----> for RAG-based applications

Now on paper, this sounds useful.
But in practice, it became confusing.
As a developer ,you always had to think: which chain shuld i use for this problem and sometime your case didn't fit perfectly into any one chain.
so you either had:
    .force your problem into an existing chain
    .or combine multiple chains in a complicated way
this made things harder insted of easier.on top of that, chains were not very flexible.
SO DEAL WITH THIS TYPE OF SITUATION THEY INTRODUCE RUNNABLES


In [ ]:
# RUNNABLES
Now instead of treating all of these differently, Langchain introduced a unified solution
Everything is a Runnable.
That means:
your prompts , model, output, python function can be runnable.
so instead of thinking in terms of different types of chains, now you just think in terms of runnables that can be connected together.
** Actual Runnables**
LLM | Prompt Template | Output Parser | Retriever | Tools --> basically they do actual jobs


In [ ]:
Core Concepts
Standard Interface: Every Runnable implements consistent methods like invoke() (single input), batch() (multiple inputs), and stream() (chunked output).
Composability: You can "snap" different Runnables together using the pipe operator (|) to create complex AI workflows, similar to building with LEGO blocks.
Recursive Nature: Once you chain several Runnables together, the resulting "chain" is itself a Runnable, allowing you to nest workflows within other workflows.
Common Types:
***RunnableParallel: Executes multiple tasks simultaneously using the same initial input and returns a dictionary where each key corresponds
to a task's output.
***RunnablePassthrough: Acts as an "identity" function that passes data through unchanged; it is primarily used to preserve original inputs
so they can be accessed later in a chain.
***RunnableLambda: A wrapper that converts any standard Python function or "callable" into a Runnable, allowing you to inject custom logic
into a chain.
***RunnableBranch: Implements conditional "if-else" logic, routing input to different Runnables based on specific conditions you define

In [ ]:
# sequence-runnable

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. Initialize the model with the API key directly
# Replace 'your_api_key_here' with the key you created in GroqCloud
llm = ChatGroq(
    groq_api_key='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', 
    model_name="llama-3.3-70b-versatile",
    temperature=0.5
)

# 2. Define your prompt
prompt = ChatPromptTemplate.from_template("Explain {topic} in two sentence.")

parser = StrOutputParser()

# 3. Create and run the chain
chain = prompt | llm | parser

response = chain.invoke({"topic": "cricket"})
print(response)

In [11]:
# parallel runnables
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel  # Import RunnableParallel

# 1. Initialize the model with the API key directly
model = ChatGroq(
    groq_api_key='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', 
    model_name="llama-3.3-70b-versatile",
    temperature=0.5
)

parser = StrOutputParser()

# two different prompts
short_prompt = ChatPromptTemplate.from_template("Explain {topic} in two sentence")
detailed_prompt = ChatPromptTemplate.from_template("give {topic} name")

# 3. Create and run the chain using RunnableParallel
chain = RunnableParallel(  
    short=short_prompt | model | parser,
    detailed=detailed_prompt | model | parser
)

response = chain.invoke({"topic": "cricket"})
print(response['short'])
print(response['detailed'])

Cricket is a team sport played with a bat and ball, where two teams of 11 players each take turns to score runs by hitting the ball with a bat and running between two sets of three stumps (wickets) while the opposing team tries to stop them. The team with the most runs at the end of the game wins, with the game typically divided into innings, where each team gets a chance to bat and bowl (throw the ball) in an attempt to outscore their opponents.
Here are some popular cricket team names:

1. **Mighty XI**
2. **Cricket Crushers**
3. **Pitch Perfect**
4. **Sixers Squad**
5. **The Bowling Brigade**
6. **The Batting Masters**
7. **Field Fury**
8. **The Cricket Kings**
9. **Spin Doctors**
10. **The Wicket Warriors**

Which one do you like the most? Or would you like me to suggest more names?


In [12]:
# what if we want different topics for different chain ----> use lambda

# parallel runnables
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel , RunnableLambda

# 1. Initialize the model with the API key directly
model = ChatGroq(
    groq_api_key='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', 
    model_name="llama-3.3-70b-versatile",
    temperature=0.5
)

parser = StrOutputParser()

# two different prompts
short_prompt = ChatPromptTemplate.from_template("Explain {topic} in two sentence")
detailed_prompt = ChatPromptTemplate.from_template("give {topic} different name")

# 3. Create and run the chain using RunnableParallel
chain = RunnableParallel(  
    short= RunnableLambda(lambda x:x['short'])|short_prompt | model | parser,
    detailed= RunnableLambda(lambda x:x['detailed'])|detailed_prompt | model | parser
)

response = chain.invoke({
    "short":{"topic":"cricket"},
    "detailed":{"topic":"biology"}
})
print(response['short'])
print(response['detailed'])

Cricket is a team sport played with a bat and ball, where two teams of 11 players each take turns to score runs by hitting the ball and running between two sets of three stumps (wickets) while the opposing team tries to stop them. The team with the most runs at the end of the game, which can range from a few hours to several days, is declared the winner, with various rules and strategies involved to outmaneuver the opposing team.
Instead of "biology," let's call it:

1. **Vitalis**: Derived from the Latin word "vitalis," meaning "of or pertaining to life."
2. **Ecozoa**: A combination of "eco" (environment) and "zoa" (life), emphasizing the connection between living organisms and their surroundings.
3. **Biogenics**: Suggesting the study of the generation and diversity of life.
4. **Organika**: Focusing on the complex organization and structure of living organisms.
5. **Zoologia**: While this name is similar to "zoology," it could be used to describe the broader field of biology, encom

In [15]:
# runnable passthrough
# parallel runnables
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel,RunnablePassthrough  # Import RunnableParallel

# 1. Initialize the model with the API key directly
model = ChatGroq(
    groq_api_key='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', 
    model_name="llama-3.3-70b-versatile",
    temperature=0.5
)

parser = StrOutputParser()

# two different prompts
code_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a code generator"),
    ("human", "{topic}")
])

explain_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant who explains code in simple terms"),
    ("human", "Explain the following code in simple words:\n{code}")
])

seq= code_prompt | model | parser
seq2= RunnableParallel(
    code=RunnablePassthrough(), 
    explanation=explain_prompt|model|parser  
)

# 3. Create and run the chain using RunnableParallel
chain =seq|seq2

response = chain.invoke({"topic": "write a code in python of palindrome"})
print(response['code'])
print(response['explanation'])

**Palindrome Checker in Python**

This code checks if a given string is a palindrome or not.

**Code**
------
```python
def is_palindrome(s):
    """
    Checks if a given string is a palindrome.

    Args:
        s (str): The input string.

    Returns:
        bool: True if the string is a palindrome, False otherwise.
    """
    s = ''.join(c for c in s if c.isalnum()).lower()  # remove non-alphanumeric characters and convert to lowercase
    return s == s[::-1]  # check if the string is equal to its reverse

# Example usage:
def main():
    strings = ["radar", "hello", "A man, a plan, a canal: Panama"]
    for s in strings:
        print(f"'{s}' is a palindrome: {is_palindrome(s)}")

if __name__ == "__main__":
    main()
```

**Explanation**
---------------

1. The `is_palindrome` function takes a string `s` as input.
2. It removes non-alphanumeric characters from the string and converts it to lowercase to ensure the comparison is case-insensitive.
3. It checks if the resulting st